# Step 7: Exploratory Data Analysis (EDA)

## 1. Executive Summary & Objectives
Exploratory Data Analysis provides analytical insights into feature distributions, binary prevalence contrasts, correlation structures, and target relationships across the 30,000 dataset samples.

## Key Analytical Questions Answered:
1. **Target Distribution**: How balanced are the class labels in the dataset?
2. **Continuous Feature Distributions**: Do URL length, digit counts, and character ratios differ significantly between Phishing and Legitimate classes?
3. **Binary Feature Prevalence**: Which binary flags provide the strongest discriminative contrast?
4. **Correlation & Association Analysis**: What features exhibit the highest correlation with `is_phishing`?
5. **Deployability & Feature Feasibility**: How do URL-string derivable features (`FS-URL`) compare in signal strength against page content features (`FS-REF`)?

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append("..")
from src.config import RAW_DATA_PATH, RAW_TARGET_COL, DERIVED_TARGET_COL, FIGURES_DIR, TABLES_DIR

# Load raw dataset and create target mapping
df = pd.read_csv(RAW_DATA_PATH)
df[DERIVED_TARGET_COL] = (df[RAW_TARGET_COL] == 0).astype("int8")
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")


Dataset loaded: 30,000 rows x 57 columns


## 2. Target Class Distribution Analysis
Target distribution is evaluated to confirm class proportions before modeling.

In [7]:
counts = df[DERIVED_TARGET_COL].value_counts()
pcts = df[DERIVED_TARGET_COL].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    "Target Class": ["Legitimate (0)", "Phishing (1)"],
    "Sample Count": [counts[0], counts[1]],
    "Percentage": [f"{pcts[0]:.2f}%", f"{pcts[1]:.2f}%"]
})
print(target_summary.to_string(index=False))
print(f"\nClass Imbalance Ratio (Legitimate / Phishing): {counts[0]/counts[1]:.3f}")


  Target Class  Sample Count Percentage
Legitimate (0)         17157     57.19%
  Phishing (1)         12843     42.81%

Class Imbalance Ratio (Legitimate / Phishing): 1.336


## 3. URL Length & Continuous Feature Analysis
Comparing `URLLength` across target classes using log-scale distributions.

In [8]:
stats_df = df.groupby(DERIVED_TARGET_COL)['URLLength'].agg(['count', 'mean', 'std', 'median', 'min', 'max'])
stats_df.index = ['Legitimate (0)', 'Phishing (1)']
print("URL Length Summary Statistics:")
print(stats_df.to_string())


URL Length Summary Statistics:
                count       mean        std  median  min   max
Legitimate (0)  17157  26.267238   4.825853    26.0   16    51
Phishing (1)    12843  46.286460  78.218665    34.0   13  4274


## 4. Binary Feature Prevalence Comparison
Binary features represent key syntactic and structural flags. Lower prevalence of `IsHTTPS` and higher prevalence of `IsDomainIP` are key indicators.

In [9]:
binary_cols = ['IsHTTPS', 'IsDomainIP', 'HasObfuscation', 'HasTitle', 'HasFavicon', 
               'Robots', 'IsResponsive', 'HasDescription', 'HasExternalFormSubmit', 
               'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField']

bin_summary = []
for col in binary_cols:
    prev_legit = df[df[DERIVED_TARGET_COL] == 0][col].mean() * 100
    prev_phish = df[df[DERIVED_TARGET_COL] == 1][col].mean() * 100
    bin_summary.append({
        'Feature': col,
        'Legitimate_%': f"{prev_legit:.2f}%",
        'Phishing_%': f"{prev_phish:.2f}%",
        'Contrast_Diff_%': f"{abs(prev_phish - prev_legit):.2f}%"
    })

bin_prev_df = pd.DataFrame(bin_summary)
print("Binary Feature Prevalence Contrast Table:")
print(bin_prev_df.to_string(index=False))


Binary Feature Prevalence Contrast Table:
              Feature Legitimate_% Phishing_% Contrast_Diff_%
              IsHTTPS      100.00%     48.74%          51.26%
           IsDomainIP        0.00%      0.55%           0.55%
       HasObfuscation        0.00%      0.48%           0.48%
             HasTitle       99.88%     67.20%          32.68%
           HasFavicon       56.29%      8.71%          47.58%
               Robots       41.11%      6.32%          34.79%
         IsResponsive       85.21%     31.60%          53.61%
       HasDescription       73.47%      4.13%          69.33%
HasExternalFormSubmit        7.30%      0.42%           6.88%
         HasSocialNet       78.98%      0.51%          78.47%
      HasSubmitButton       66.05%      8.28%          57.77%
      HasHiddenFields       58.52%      9.34%          49.18%
     HasPasswordField       14.14%      5.45%           8.69%


## 5. Feature Correlation Analysis
Pearson correlations with `is_phishing` highlight top linear predictors.

In [10]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['label', 'is_phishing'], errors='ignore')
corr_series = numeric_df.apply(lambda col: col.corr(df['is_phishing'])).abs().sort_values(ascending=False)

top_10 = pd.DataFrame({
    'Feature': corr_series.head(10).index,
    'Abs_Correlation': corr_series.head(10).values
})
print("Top 10 Features Correlated with Target (is_phishing):")
print(top_10.to_string(index=False))


Top 10 Features Correlated with Target (is_phishing):
              Feature  Abs_Correlation
   URLSimilarityIndex         0.858887
         HasSocialNet         0.779851
     HasCopyrightInfo         0.748543
       HasDescription         0.691494
              IsHTTPS         0.612821
DomainTitleMatchScore         0.584776
      HasSubmitButton         0.580503
         IsResponsive         0.547226
   URLTitleMatchScore         0.540495
               NoOfJS         0.532928
